# FedFlower Phase 4 — Model Conversion to PyTorch Mobile

> **Before running:** Go to `Runtime → Change Runtime Type → T4 GPU → Save`

This notebook converts `best_model.pth` to a TorchScript traced model (`flower_traced.pt`) for use with the Android PyTorch Mobile runtime.

**Steps:**
1. Upload `best_model.pth`
2. Freeze all BatchNorm layers explicitly
3. `model.eval()`
4. `torch.jit.trace` with dummy input `[1, 3, 224, 224]`
5. Save and download `flower_traced.pt`

## Cell 1 — Install PyTorch & Imports

In [ ]:
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu118 -q
print('✅ PyTorch ready')

import torch
import torch.nn as nn
import torchvision.models as models
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## Cell 2 — Upload best_model.pth

When the file picker appears, select `best_model.pth` from your laptop.

In [ ]:
from google.colab import files
print('Upload best_model.pth from your laptop...')
uploaded = files.upload()
assert 'best_model.pth' in uploaded, '❌ Wrong filename — must be best_model.pth'
print(f'✅ Uploaded best_model.pth ({os.path.getsize("best_model.pth")/1e6:.1f} MB)')

## Cell 3 — Define FlowerCNN & Load Weights

In [ ]:
class FlowerCNN(nn.Module):
    def __init__(self, num_classes=102):
        super().__init__()
        self.backbone = models.resnet50(weights='IMAGENET1K_V2')
        for name, param in self.backbone.named_parameters():
            if 'layer4' not in name and 'fc' not in name:
                param.requires_grad = False
        in_f = self.backbone.fc.in_features
        self.backbone.fc = nn.Sequential(
            nn.Linear(in_f, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
            nn.Dropout(0.4), nn.Linear(512, num_classes))
    def forward(self, x): return self.backbone(x)

model = FlowerCNN(num_classes=102)
model.load_state_dict(torch.load('best_model.pth', map_location='cpu'))
print('✅ Weights loaded')

## Cell 4 — Freeze BatchNorm & Set eval()

BatchNorm layers must be **explicitly frozen** before tracing. Without this, batch statistics computed during tracing can shift at inference time on Android, causing incorrect predictions.

`model.eval()` alone is not enough — the explicit loop ensures every BN layer is locked.

In [ ]:
# Step 1: set the whole model to eval mode (disables Dropout, switches BN to use running stats)
model.eval()

# Step 2: explicitly freeze every BatchNorm layer so tracing captures fixed statistics
bn_count = 0
for m in model.modules():
    if isinstance(m, (nn.BatchNorm1d, nn.BatchNorm2d)):
        m.eval()
        bn_count += 1

print(f'✅ model.eval() set')
print(f'✅ Explicitly froze {bn_count} BatchNorm layers (BatchNorm1d + BatchNorm2d)')
print('   BatchNorm will use stored running_mean/running_var during Android inference')

## Cell 5 — Trace & Save as flower_traced.pt

`torch.jit.trace` records the exact computation graph for the dummy input shape `[1, 3, 224, 224]`. The output file is loaded directly by the Android PyTorch Mobile runtime.

In [ ]:
dummy = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    traced_model = torch.jit.trace(model, dummy)

traced_model.save('flower_traced.pt')

size_mb = os.path.getsize('flower_traced.pt') / 1e6
print(f'✅ Saved flower_traced.pt ({size_mb:.1f} MB)')

## Cell 6 — Verify Traced Model Output Matches Original

Both models should produce identical logits for the same input (max diff < 0.001).

In [ ]:
import numpy as np

test_input = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    orig_out   = model(test_input).numpy()
    traced_out = traced_model(test_input).numpy()

max_diff = np.max(np.abs(orig_out - traced_out))
print(f'Max output diff (orig vs traced): {max_diff:.8f}')
assert max_diff < 0.001, f'❌ Outputs diverged too much: {max_diff}'
print('✅ Traced model matches original — safe to deploy')

orig_cls   = int(np.argmax(orig_out))
traced_cls = int(np.argmax(traced_out))
print(f'   Predicted class: {orig_cls + 1} (orig) vs {traced_cls + 1} (traced) — match: {orig_cls == traced_cls}')

## Cell 7 — Download flower_traced.pt ⬇️

After downloading, copy this file into `android/app/src/main/assets/` before building the Android app.

In [ ]:
from google.colab import files
print(f'Downloading flower_traced.pt ({os.path.getsize("flower_traced.pt")/1e6:.1f} MB)...')
files.download('flower_traced.pt')
print('✅ Downloaded flower_traced.pt')
print('Next step: copy it to  android/app/src/main/assets/flower_traced.pt')